# feature engineering 2.0
- following from second meeting with Greenberg 
- feedback included:
-	CARD might be better because we know they have diagnosis 
-	Rerun and make a more specific and comprehensive list of features and ensure that you know what each feature is. 
o	When rerunning if you want to include individual items do it but not domains (not sure what this meant but figure it out and try it)
o	Surprised AQ and Sex not big predictors 
-	Rerun but target group excludes people ‘with diagnosis’ but score below threshold of AQ
-	Rerun with AQ as target Var e.g., 0 class below a 6? and target case 6 and above?
-	Rerun but with my own PCA on each of the questions 
-	Rerun with AQ cut off as target var 
-	Also ‘other’ could have been a text option so maybe look into this?
-	Oh and email don’t use linked in lol 


# notebook uses beginning of feature_engineering 1.0 to set up the same dataset



In [ ]:
import pandas as pd
import numpy as np 
from sklearn.feature_selection import VarianceThreshold
# Add these imports to your notebook
from lightgbm import LGBMClassifier
from sklearn.ensemble import StackingClassifier, VotingClassifier

# load matched data 
df = pd.read_csv('data/processed/data_c4_matched_balanced.csv')
import os

# Create the directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)


# 1. feature creation 

# age group bins
df['age_group'] = pd.cut(df['age'], bins=[0, 18, 30, 45, 60, 100], labels=['0-18', '19-30', '31-45', '46-60', '61+'])

#non linear transformation 
df['log_aq_total'] = np.log1p(df['aq_total'])
df['sqrt_age'] = np.sqrt(df['age'])

# interaction terms
df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']
df['sqp_aq_interaction'] = df['spq_total'] * df['aq_total']
df['age_x_eq'] = df['age'] * df['eq_total']

# questionnaire score ratios 
df['aq_spq_ratio'] = df['aq_total'] / (df['spq_total'] + 1e-8)
df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)

#boolean: high aq (above 1 std)
df['high_aq'] = (df['aq_total'] > df['aq_total'].mean() + df['aq_total'].std()).astype(int)

# 2. feature reduction/selection

# remove highly correlated features 
# Only use numeric columns for correlation
numeric_cols = df.drop(columns=['autism_target']).select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df = df.drop(columns=to_drop)

# drop low variance features 
# Only apply VarianceThreshold to numeric columns
feature_cols = df.drop(columns=['autism_target']).select_dtypes(include=[np.number]).columns
selector = VarianceThreshold(threshold=0.1)
selector.fit(df[feature_cols])
low_variance_cols = feature_cols[~selector.get_support()]
df = df.drop(columns=low_variance_cols)

# 3. one-hot encode new categorical features 
df = pd.get_dummies(df, columns=['age_group'], drop_first=True)

# 4. save engineered dataset 
df.to_csv('../data/processed/data_c4_balanced_fe.csv', index=False)

print("feature engineering complete. new shape:", df.shape)
print("columns:", df.columns.tolist())

# baseline models

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# df load data
df = pd.read_csv('data/processed/data_c4_balanced_fe.csv')
x = df.drop(columns=['autism_target'])
y = df['autism_target']

# Handle missing values
imputer = SimpleImputer(strategy='mean')
x_imputed = pd.DataFrame(imputer.fit_transform(x), columns=x.columns)

# Scale features for Logistic Regression only
scaler = StandardScaler()
x_scaled = pd.DataFrame(scaler.fit_transform(x_imputed), columns=x_imputed.columns)

x_train, x_val, y_train, y_val = train_test_split(x_imputed, y, stratify=y, test_size=0.2, random_state=42)
x_train_scaled, x_val_scaled, y_train_scaled, y_val_scaled = train_test_split(x_scaled, y, stratify=y, test_size=0.2, random_state=42)

# Store all models and their results
models = {}
results = {}

# 1. Logistic Regression
logreg = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
logreg.fit(x_train_scaled, y_train_scaled)
logreg_pred = logreg.predict(x_val_scaled)
logreg_proba = logreg.predict_proba(x_val_scaled)[:, 1]
logreg_auc = roc_auc_score(y_val_scaled, logreg_proba)
logreg_f1 = f1_score(y_val_scaled, logreg_pred)

models['logreg'] = logreg
results['logreg'] = {'auc': logreg_auc, 'f1': logreg_f1, 'model': logreg, 'data': 'scaled'}

print("Logistic Regression:")
print(f"ROC-AUC: {logreg_auc:.4f}")
print(f"F1-Score: {logreg_f1:.4f}")
print(classification_report(y_val_scaled, logreg_pred))

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(x_train, y_train)
rf_pred = rf.predict(x_val)
rf_proba = rf.predict_proba(x_val)[:, 1]
rf_auc = roc_auc_score(y_val, rf_proba)
rf_f1 = f1_score(y_val, rf_pred)

models['rf'] = rf
results['rf'] = {'auc': rf_auc, 'f1': rf_f1, 'model': rf, 'data': 'unscaled'}

print("\nRandom Forest:")
print(f"ROC-AUC: {rf_auc:.4f}")
print(f"F1-Score: {rf_f1:.4f}")
print(classification_report(y_val, rf_pred))

# 3. XGBoost
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(x_train, y_train)
xgb_pred = xgb.predict(x_val)
xgb_proba = xgb.predict_proba(x_val)[:, 1]
xgb_auc = roc_auc_score(y_val, xgb_proba)
xgb_f1 = f1_score(y_val, xgb_pred)

models['xgb'] = xgb
results['xgb'] = {'auc': xgb_auc, 'f1': xgb_f1, 'model': xgb, 'data': 'unscaled'}

print("\nXGBoost:")
print(f"ROC-AUC: {xgb_auc:.4f}")
print(f"F1-Score: {xgb_f1:.4f}")
print(classification_report(y_val, xgb_pred))

# 4. LightGBM
lgb = LGBMClassifier(random_state=42, verbose=-1)
lgb.fit(x_train, y_train)
lgb_pred = lgb.predict(x_val)
lgb_proba = lgb.predict_proba(x_val)[:, 1]
lgb_auc = roc_auc_score(y_val, lgb_proba)
lgb_f1 = f1_score(y_val, lgb_pred)

models['lgb'] = lgb
results['lgb'] = {'auc': lgb_auc, 'f1': lgb_f1, 'model': lgb, 'data': 'unscaled'}

print("\nLightGBM:")
print(f"ROC-AUC: {lgb_auc:.4f}")
print(f"F1-Score: {lgb_f1:.4f}")
print(classification_report(y_val, lgb_pred))

# 5. Gradient Boosting
gb = GradientBoostingClassifier(random_state=42)
gb.fit(x_train, y_train)
gb_pred = gb.predict(x_val)
gb_proba = gb.predict_proba(x_val)[:, 1]
gb_auc = roc_auc_score(y_val, gb_proba)
gb_f1 = f1_score(y_val, gb_pred)

models['gb'] = gb
results['gb'] = {'auc': gb_auc, 'f1': gb_f1, 'model': gb, 'data': 'unscaled'}

print("\nGradient Boosting:")
print(f"ROC-AUC: {gb_auc:.4f}")
print(f"F1-Score: {gb_f1:.4f}")
print(classification_report(y_val, gb_pred))

# 6. Extra Trees (faster than RF, often better performance)
et = ExtraTreesClassifier(n_estimators=100, random_state=42)
et.fit(x_train, y_train)
et_pred = et.predict(x_val)
et_proba = et.predict_proba(x_val)[:, 1]
et_auc = roc_auc_score(y_val, et_proba)
et_f1 = f1_score(y_val, et_pred)

models['et'] = et
results['et'] = {'auc': et_auc, 'f1': et_f1, 'model': et, 'data': 'unscaled'}

print("\nExtra Trees:")
print(f"ROC-AUC: {et_auc:.4f}")
print(f"F1-Score: {et_f1:.4f}")
print(classification_report(y_val, et_pred))

# 7. AdaBoost (fast, often good performance)
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(x_train, y_train)
ada_pred = ada.predict(x_val)
ada_proba = ada.predict_proba(x_val)[:, 1]
ada_auc = roc_auc_score(y_val, ada_proba)
ada_f1 = f1_score(y_val, ada_pred)

models['ada'] = ada
results['ada'] = {'auc': ada_auc, 'f1': ada_f1, 'model': ada, 'data': 'unscaled'}

print("\nAdaBoost:")
print(f"ROC-AUC: {ada_auc:.4f}")
print(f"F1-Score: {ada_f1:.4f}")
print(classification_report(y_val, ada_pred))

# Find best model
best_model_name = max(results.keys(), key=lambda k: results[k]['auc'])
best_model = results[best_model_name]['model']
best_auc = results[best_model_name]['auc']
best_f1 = results[best_model_name]['f1']

print(f"\n{'='*50}")
print(f"BEST MODEL: {best_model_name.upper()}")
print(f"ROC-AUC: {best_auc:.4f}")
print(f"F1-Score: {best_f1:.4f}")
print(f"{'='*50}")

# Store best model for later use
best_model_name_final = best_model_name

# threshold moving
- for best model only 

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score, classification_report, roc_auc_score

# Get predictions from best model
if results[best_model_name_final]['data'] == 'scaled':
    best_probs = best_model.predict_proba(x_val_scaled)[:, 1]
    best_x_val = x_val_scaled
    best_y_val = y_val_scaled
else:
    best_probs = best_model.predict_proba(x_val)[:, 1]
    best_x_val = x_val
    best_y_val = y_val

# Find best threshold for F1 score
prec, rec, thresholds = precision_recall_curve(best_y_val, best_probs)
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
best_thresh = thresholds[np.argmax(f1_scores)]

print(f"Best Model ({best_model_name_final.upper()}) - Best threshold for F1: {best_thresh:.3f}")

# Evaluate at best threshold
best_pred_thresh = (best_probs >= best_thresh).astype(int)
print(f"\n{best_model_name_final.upper()} validation set performance at best threshold:")
print(classification_report(best_y_val, best_pred_thresh))
print(f"F1 at best threshold: {f1_score(best_y_val, best_pred_thresh):.4f}")
print(f"ROC-AUC: {roc_auc_score(best_y_val, best_probs):.4f}")

# Store best model and data for feature importance
best_model_final = best_model
best_x_val_final = best_x_val
best_y_val_final = best_y_val

# feature importance
- on best model only

In [ ]:
# Feature importance analysis for the BEST performing model only
import pandas as pd
from sklearn.inspection import permutation_importance

print(f"Feature Importance Analysis for {best_model_name_final.upper()} (Best Model)")
print("="*60)

# Get feature importance based on model type
if hasattr(best_model_final, 'feature_importances_'):
    # Tree-based models (RF, XGB, LGB, GB)
    importances = pd.Series(best_model_final.feature_importances_, index=x_train.columns)
    print(f"\nTop 20 features by {best_model_name_final.upper()} importance:")
    print(importances.sort_values(ascending=False).head(20))
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'Feature': x_train.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
else:
    # Linear models (LogReg, SVM)
    if hasattr(best_model_final, 'coef_'):
        # Logistic Regression
        importances = pd.Series(np.abs(best_model_final.coef_[0]), index=x_train.columns)
    else:
        # SVM or other models
        importances = pd.Series(np.zeros(len(x_train.columns)), index=x_train.columns)
    
    print(f"\nTop 20 features by {best_model_name_final.upper()} coefficients (absolute values):")
    print(importances.sort_values(ascending=False).head(20))
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'Feature': x_train.columns,
        'Coefficient': importances
    }).sort_values('Coefficient', ascending=False)

# Permutation importance (more robust, works for all models)
print(f"\nComputing permutation importance for {best_model_name_final.upper()}...")
perm_importance = permutation_importance(
    best_model_final, 
    best_x_val_final, 
    best_y_val_final, 
    n_repeats=5, 
    random_state=42
)

perm_importance_df = pd.DataFrame({
    'Feature': x_train.columns,
    'Permutation_importance': perm_importance.importances_mean
}).sort_values('Permutation_importance', ascending=False)

print(f"\nTop 20 features by Permutation importance ({best_model_name_final.upper()}):")
print(perm_importance_df.head(20))

# Save both importance measures
print(f"\nSaving feature importance results for {best_model_name_final.upper()}...")
importance_df.to_csv(f'feature_importance_{best_model_name_final}.csv', index=False)
perm_importance_df.to_csv(f'permutation_importance_{best_model_name_final}.csv', index=False)

print(f"\nFeature importance analysis complete for {best_model_name_final.upper()}")
print(f"Results saved to: feature_importance_{best_model_name_final}.csv")
print(f"Permutation importance saved to: permutation_importance_{best_model_name_final}.csv")

# rerun models with individual items only 
- result = reduced performance

In [ ]:
# Remove total scores and domain aggregations
features_to_drop = ['aq_total', 'spq_total', 'eq_total', 'sqr_total', 'd_score']
x_individual = x.drop(columns=features_to_drop)

# Split the data again
x_train_ind, x_val_ind, y_train_ind, y_val_ind = train_test_split(
    x_individual, y, test_size=0.2, random_state=42, stratify=y
)

# Train Random Forest with individual items
rf_ind = RandomForestClassifier(random_state=42)
rf_ind.fit(x_train_ind, y_train_ind)
rf_ind_probs = rf_ind.predict_proba(x_val_ind)[:, 1]

# Train XGBoost with individual items
xgb_ind = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_ind.fit(x_train_ind, y_train_ind)
xgb_ind_probs = xgb_ind.predict_proba(x_val_ind)[:, 1]

# Find best thresholds for F1 score
rf_ind_thresholds = np.linspace(0, 1, 100)
rf_ind_f1_scores = [f1_score(y_val_ind, (rf_ind_probs >= t).astype(int)) for t in rf_ind_thresholds]
rf_ind_best_thresh = rf_ind_thresholds[np.argmax(rf_ind_f1_scores)]

xgb_ind_thresholds = np.linspace(0, 1, 100)
xgb_ind_f1_scores = [f1_score(y_val_ind, (xgb_ind_probs >= t).astype(int)) for t in xgb_ind_thresholds]
xgb_ind_best_thresh = xgb_ind_thresholds[np.argmax(xgb_ind_f1_scores)]

# Evaluate models with individual items
print("\n--- Models with individual items only ---")
print(f"Random Forest - Best threshold for F1: {rf_ind_best_thresh:.3f}")
rf_ind_pred_thresh = (rf_ind_probs >= rf_ind_best_thresh).astype(int)
print("Random Forest validation set performance:")
print(classification_report(y_val_ind, rf_ind_pred_thresh))
print(f"F1 at best threshold: {f1_score(y_val_ind, rf_ind_pred_thresh):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val_ind, rf_ind_probs):.3f}")

print(f"\nXGBoost - Best threshold for F1: {xgb_ind_best_thresh:.3f}")
xgb_ind_pred_thresh = (xgb_ind_probs >= xgb_ind_best_thresh).astype(int)
print("XGBoost validation set performance:")
print(classification_report(y_val_ind, xgb_ind_pred_thresh))
print(f"F1 at best threshold: {f1_score(y_val_ind, xgb_ind_pred_thresh):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val_ind, xgb_ind_probs):.3f}")

# Keep only individual items and engineered features
# This will test if individual questions are more predictive than totals

# rerun model with only total scores 
- result = reduced performance

In [ ]:
# Keep only total scores and engineered features
individual_cols = [col for col in x.columns if any(col.startswith(prefix) for prefix in ['aq_', 'spq_', 'eq_', 'sqr_'])]
x_totals_only = x.drop(columns=individual_cols)

# Split the data
x_train_tot, x_val_tot, y_train_tot, y_val_tot = train_test_split(
    x_totals_only, y, test_size=0.2, random_state=42, stratify=y
)

# Train Random Forest with total scores
rf_tot = RandomForestClassifier(random_state=42)
rf_tot.fit(x_train_tot, y_train_tot)
rf_tot_probs = rf_tot.predict_proba(x_val_tot)[:, 1]

# Train XGBoost with total scores
xgb_tot = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_tot.fit(x_train_tot, y_train_tot)
xgb_tot_probs = xgb_tot.predict_proba(x_val_tot)[:, 1]

# Find best thresholds for F1 score
rf_tot_thresholds = np.linspace(0, 1, 100)
rf_tot_f1_scores = [f1_score(y_val_tot, (rf_tot_probs >= t).astype(int)) for t in rf_tot_thresholds]
rf_tot_best_thresh = rf_tot_thresholds[np.argmax(rf_tot_f1_scores)]

xgb_tot_thresholds = np.linspace(0, 1, 100)
xgb_tot_f1_scores = [f1_score(y_val_tot, (xgb_tot_probs >= t).astype(int)) for t in xgb_tot_thresholds]
xgb_tot_best_thresh = xgb_tot_thresholds[np.argmax(xgb_tot_f1_scores)]

# Evaluate models with total scores
print("\n--- Models with total scores only ---")
print(f"Random Forest - Best threshold for F1: {rf_tot_best_thresh:.3f}")
rf_tot_pred_thresh = (rf_tot_probs >= rf_tot_best_thresh).astype(int)
print("Random Forest validation set performance:")
print(classification_report(y_val_tot, rf_tot_pred_thresh))
print(f"F1 at best threshold: {f1_score(y_val_tot, rf_tot_pred_thresh):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val_tot, rf_tot_probs):.3f}")

print(f"\nXGBoost - Best threshold for F1: {xgb_tot_best_thresh:.3f}")
xgb_tot_pred_thresh = (xgb_tot_probs >= xgb_tot_best_thresh).astype(int)
print("XGBoost validation set performance:")
print(classification_report(y_val_tot, xgb_tot_pred_thresh))
print(f"F1 at best threshold: {f1_score(y_val_tot, xgb_tot_pred_thresh):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_val_tot, xgb_tot_probs):.3f}")

# This will test if aggregated scores are more predictive